# 11 — Postprocesamiento con ArcPy

Aplica el pipeline de postprocesamiento sobre el raster clasificado generado por el DLPK en ArcGIS Pro.

**Flujo:**
1. Correr inferencia en ArcGIS Pro → guardar raster clasificado `.tif`
2. Ejecutar este notebook → genera raster postprocesado
3. Cargar el resultado en ArcGIS Pro para visualización

**El script usa el Python de ArcGIS Pro** (`arcgispro-py3`) desde este entorno.

In [ ]:
import subprocess
from pathlib import Path

# Python de ArcGIS Pro
ARCGIS_PYTHON = r'C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe'
SCRIPT        = str(Path('scripts/postprocesar_arcpy.py').resolve())

print(f'Python ArcGIS Pro: {ARCGIS_PYTHON}')
print(f'Script:            {SCRIPT}')

In [ ]:
# ── Configurar rutas ──────────────────────────────────────────────────────────
# INPUT: raster clasificado generado por ArcGIS Pro con el DLPK
# Cambia esta ruta al archivo que exportaste desde ArcGIS Pro

INPUT_RASTER  = r'D:\IGAC_1K_SEGMENTATION\IGAC_1K_SEGMENTATION\results\inferencia_arcgis\tabio_clasificado.tif'
OUTPUT_RASTER = r'D:\IGAC_1K_SEGMENTATION\IGAC_1K_SEGMENTATION\results\inferencia_arcgis\tabio_clasificado_pp.tif'

# Crear carpeta de salida si no existe
Path(OUTPUT_RASTER).parent.mkdir(parents=True, exist_ok=True)
print(f'Input:  {INPUT_RASTER}')
print(f'Output: {OUTPUT_RASTER}')

In [ ]:
# ── Ejecutar postprocesamiento ────────────────────────────────────────────────
cmd = [
    ARCGIS_PYTHON, SCRIPT,
    '--input',  INPUT_RASTER,
    '--output', OUTPUT_RASTER,
]

print('Corriendo postprocesamiento...')
result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('ERRORES:')
    print(result.stderr)
else:
    print('Postprocesamiento completado exitosamente.')

In [ ]:
# ── Visualizar resultado (comparativo antes/despues) ─────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import rasterio
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print('rasterio no disponible - instala con: pip install rasterio')

if HAS_RASTERIO:
    import sys, os
    sys.path.insert(0, os.path.abspath('..'))
    from scripts.utils import N_CLASSES, CLASS_NAMES, CLASS_COLORS

    CMAP = np.zeros((N_CLASSES, 3), dtype=np.uint8)
    for cls_id, rgb in CLASS_COLORS.items():
        if cls_id < N_CLASSES:
            CMAP[cls_id] = rgb

    def to_rgb(mask):
        return CMAP[np.clip(mask, 0, N_CLASSES-1)]

    def read_raster(path, max_px=3000):
        with rasterio.open(path) as src:
            data = src.read(1)
        h, w = data.shape
        scale = min(max_px / max(h, w), 1.0)
        if scale < 1.0:
            nh, nw = int(h*scale), int(w*scale)
            ri = (np.arange(nh)*h/nh).astype(int)
            ci = (np.arange(nw)*w/nw).astype(int)
            data = data[np.ix_(ri, ci)]
        return data

    antes   = read_raster(INPUT_RASTER)
    despues = read_raster(OUTPUT_RASTER)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), dpi=120)
    axes[0].imshow(to_rgb(antes));   axes[0].set_title('Sin PP',   fontsize=13, fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(to_rgb(despues)); axes[1].set_title('Con PP',   fontsize=13, fontweight='bold'); axes[1].axis('off')

    legend = [mpatches.Patch(color=np.array(CLASS_COLORS[i])/255, label=CLASS_NAMES[i])
              for i in range(1, N_CLASSES)]
    fig.legend(handles=legend, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout()
    plt.savefig(str(Path(OUTPUT_RASTER).parent / 'comparativo_pp.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Comparativo guardado.')